# 02 · Preprocessing
Cleaning (NaN/inf, conflicting labels, duplicates), benign capping, rare-class handling and the train/test split. The result is cached in `data/processed/` and reused by every later step.

In [ ]:
%cd -q ..
# notebooks run from the project root so relative config paths resolve

In [ ]:
import logging
import pandas as pd
from IPython.display import Image, display

from ids import experiments as ex
from ids.config import load_config, results_dir

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(name)s: %(message)s", datefmt="%H:%M:%S")
pd.set_option("display.max_columns", 50)

# Use configs/smoke.yaml to try a notebook on synthetic data (python -m ids.synthetic first)
cfg = load_config("configs/default.yaml")
OUT = results_dir(cfg)

In [ ]:
P = ex.prepare(cfg)          # add force=True to rebuild after changing the config
pd.Series(P.meta["clean"]).drop("family_counts")

In [ ]:
pd.DataFrame({"train": P.meta["train_counts"], "test": P.meta["test_counts"]}).sort_values("train", ascending=False)

Families removed from the **multi-class** task because they are too small to evaluate (they remain in the binary and leave-one-attack-out experiments):

In [ ]:
P.meta["rare_classes_excluded"]

### Leakage check: time-block split
Run the same pipeline with `split.strategy: time_block` (flows grouped into 5-minute buckets that never straddle the split). A large score drop vs. the random split means the random split was leaking near-duplicate neighbouring flows.

```bash
ids all --split time_block
```